In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict, List, Annotated
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
import os

load_dotenv()

In [ ]:
llm = ChatGroq(model="llama-3.1-8b-instant",
    temperature=0.7)

In [ ]:
@tool
def get_stock_price(symbol:str)->str:
    """
    Return the current stock price for the symbol received in params.
    """
    price={
        "AAPL": 150.00,
        "GOOG": 2800.00,
        "MSFT": 300.00,
        "AMZN": 3500.00
    }.get(symbol, 0.0)

    return f"The current stock price of {symbol} is ${price:.2f}"

tools=[get_stock_price]

llm_with_tools=llm.bind_tools(tools)

In [ ]:
class State(TypedDict):
    messages: Annotated[List,add_messages]

def chat_model(state:State)->State:
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

graph = StateGraph(State)
graph.add_node("chat_model",chat_model)
graph.add_node("get_stock_price",ToolNode(tools))

graph.add_edge(START,"chat_model")
graph.add_conditional_edges("chat_model",tools_condition)
graph.add_edge("tools","chat_model")
graph.add_edge(["chat_model","tools"],END)

graph_builder = graph.compile()

from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))